# M4.15 · Vision Transformer (ViT)

"An Image is Worth 16×16 Words" (Dosovitskiy ve ark. 2021): görüntüyü P × P'lik yamalara bölüp her yamayı doğrusal izdüşümle bir token'a çevirmek, başa öğrenilebilir [class] token'ı eklemek, öğrenilebilir 1B konum embedding'leri, Pre-LN Transformer kodlayıcı ve MLP başı. Yama sayısı, dizi uzunluğu ve ViT-B/16'nın 86M parametresi elle; yama embedding'inin adımı P olan bir konvolüsyon olduğu (unfold = conv), yüksek çözünürlükte ince ayar için konum embedding'lerinin 2B interpolasyonu, ortalama attention uzaklığı. PyTorch'ta küçük bir ViT'i CNN ile az veride karşılaştırmak (endüktif yanlılık), [class] token vs GAP ve konum embedding'siz ViT. Ekler: 2B sin-cos konumlar, permütasyon eşdeğerliği, Swin'in pencere attention'ı.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-15-vit.html](https://m-gonul.github.io/pytorch-lab/m4-15-vit.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_15_vit.py
CHECKS = {
 "b": "from labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def t():\n        ensure(need(ns, \"n_patches\")(224, 224, 16) == 196 and ns[\"n_patches\"](384, 384, 16) == 576, \"yama sayısı\")\n        ensure(need(ns, \"vit_params\")(768, 12, 3072, 16, 3, 224, 1000) == 86567656, \"ViT-B/16\")\n        img = [[r * 4 + c for c in range(4)] for r in range(4)]\n        ensure(need(ns, \"patchify\")(img, 2) == [[0, 1, 4, 5], [2, 3, 6, 7], [8, 9, 12, 13], [10, 11, 14, 15]], \"yamalar row-major\")\n        ensure(need(ns, \"attn_flops\")(197, 768) == 2 * 197 * 197 * 768, \"attention FLOP\")\n        rows = []\n        for name, D, L, M in ((\"ViT-B\", 768, 12, 3072), (\"ViT-L\", 1024, 24, 4096), (\"ViT-H\", 1280, 32, 5120)):\n            for P in (32, 16, 14):\n                T = ns[\"n_patches\"](224, 224, P) + 1\n                lin = L * (4 * T * D * D + 2 * T * D * M)\n                att = L * ns[\"attn_flops\"](T, D)\n                rows.append([f\"{name}/{P}\", T, f\"{ns['vit_params'](D, L, M, P, 3, 224, 1000) / 1e6:.0f} M\", f\"{(lin + att) / 1e9:.1f}\", f\"{att / (lin + att):.0%}\"])\n        R_.table(\"224 × 224'te ViT ailesi\", [\"model\", \"token\", \"parametre\", \"GMAC\", \"attention payı\"], rows,\n                 note=\"Parametre sayısı yama boyutuna neredeyse bağlı değil (yalnız yama embedding'i ve konumlar), ama hesap token sayısıyla büyür: P = 16 → 14'te token %31 artar. 224'te hesabın çoğu doğrusal katmanlarda (QKV, MLP); attention payı ancak dizi uzadıkça (yüksek çözünürlük, küçük yama) baskınlaşır.\")\n    R_.test(\"sayımlar\", t)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"patchify\", \"unpatchify\", \"patch_embed_conv\", \"conv_to_linear\", \"mean_attention_distance\")])\n    rng = np.random.default_rng(0)\n\n    def t():\n        x = rng.normal(size=(3, 8, 12))\n        z = need(ns, \"patchify\")(x, 4)\n        ensure(z.shape == (6, 48), \"şekil\")\n        close(z[4], x[:, 4:8, 4:8].transpose(1, 2, 0).ravel(), what=\"yama 4 = satır 1, sütun 1; (p1, p2, c) sırası\")\n        close(need(ns, \"unpatchify\")(z, 4, 3, 8, 12), x, what=\"ters\")\n        Wc, b = rng.normal(size=(5, 3, 4, 4)), rng.normal(size=5)\n        e1 = need(ns, \"patch_embed_conv\")(x, Wc, b, 4)\n        E = need(ns, \"conv_to_linear\")(Wc)\n        close(e1, z @ E + b, atol=1e-10, what=\"adım P'li konv. = patchify @ E\")\n        g = 14\n        I = np.eye(g * g)\n        close(need(ns, \"mean_attention_distance\")(I, g, 16), 0.0, what=\"kendine bakan baş: 0\")\n        U = np.full((g * g, g * g), 1 / (g * g))\n        local = np.zeros((g * g, g * g))\n        yy, xx = np.divmod(np.arange(g * g), g)\n        nb = (np.abs(yy[:, None] - yy[None]) <= 1) & (np.abs(xx[:, None] - xx[None]) <= 1)\n        local[nb] = 1\n        local /= local.sum(1, keepdims=True)\n        rows = [[\"yalnız kendisi\", f\"{ns['mean_attention_distance'](I, g, 16):.1f}\"], [\"3 × 3 komşuluk (CNN benzeri)\", f\"{ns['mean_attention_distance'](local, g, 16):.1f}\"], [\"düzgün (küresel)\", f\"{ns['mean_attention_distance'](U, g, 16):.1f}\"]]\n        ensure(float(rows[1][1]) < float(rows[2][1]), \"yerel < küresel\")\n        R_.table(\"Ortalama attention uzaklığı (224 × 224, P = 16, piksel)\", [\"attention deseni\", \"uzaklık\"], rows,\n                 note=\"Makale Şekil 7: ViT'in ilk katmanlarında bazı başlar zaten neredeyse tüm görüntüye bakar (~100+ piksel), bazıları yereldir (CNN'in ilk katmanlarındaki receptive field gibi); derinlikle tüm başlar küreselleşir. Bu ölçü, yerelliğin önsel olarak verilmediğini, veriden öğrenildiğini gösterir.\")\n    R_.test(\"yamalar ve attention uzaklığı\", t)\n",
 "d": "import time\nimport torch\nimport torch.nn as nn\nfrom labcheck import close, need, ensure\n\n\ndef _cnn():\n    return nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n                         nn.Flatten(), nn.Linear(64 * 16, 64), nn.ReLU(), nn.Linear(64, 10))\n\n\ndef check(ns, R_):\n    X, y = need(ns, \"digits16\")()\n    g = torch.Generator().manual_seed(0)\n    p = torch.randperm(len(X), generator=g)\n    tr, te = p[:1200], p[1200:]\n\n    def t_pe():\n        torch.manual_seed(0)\n        pe = need(ns, \"PatchEmbed\")(4, 3, 8)\n        x = torch.randn(2, 3, 16, 12)\n        a = pe(x)\n        ensure(tuple(a.shape) == (2, 12, 8), \"(B, N, D)\")\n        close(a.detach().numpy(), pe.forward_unfold(x).detach().numpy(), atol=1e-5, what=\"conv = unfold + doğrusal\")\n        v = need(ns, \"ViT\")()\n        ensure(tuple(v(X[:3]).shape) == (3, 10), \"çıkış\")\n        ensure(v.pos.shape[1] == 17 and v.cls.shape[-1] == 64, \"17 token (16 yama + [class])\")\n    R_.test(\"yama embedding'i ve model\", t_pe)\n\n    def t_train():\n        rows, acc = [], {}\n        t00 = time.perf_counter()\n        for name, mk, n, ep in ((\"CNN\", _cnn, 300, 40), (\"ViT ([class])\", lambda: ns[\"ViT\"](), 300, 40), (\"ViT, konum embedding'i yok\", lambda: ns[\"ViT\"](pos=False), 300, 40),\n                                (\"CNN\", _cnn, 1200, 20), (\"ViT ([class])\", lambda: ns[\"ViT\"](), 1200, 20), (\"ViT (GAP)\", lambda: ns[\"ViT\"](pool=\"gap\"), 1200, 20)):\n            torch.manual_seed(0)\n            m = mk()\n            a = need(ns, \"train_eval\")(m, X[tr[:n]], y[tr[:n]], X[te], y[te], ep)\n            acc[(name, n)] = a\n            rows.append([name, n, sum(q.numel() for q in m.parameters()), f\"{a:.3f}\"])\n        ensure(acc[(\"CNN\", 300)] > acc[(\"ViT ([class])\", 300)], \"az veride CNN önde\")\n        ensure(acc[(\"ViT ([class])\", 300)] > acc[(\"ViT, konum embedding'i yok\", 300)] + 0.08, \"konum embedding'i gerekli\")\n        ensure(acc[(\"ViT ([class])\", 1200)] > acc[(\"ViT ([class])\", 300)] + 0.03, \"ViT veriyle iyileşir\")\n        R_.table(f\"16 × 16 rakamlar, P = 4 ({time.perf_counter() - t00:.0f} sn)\", [\"model\", \"eğitim örneği\", \"parametre\", \"test doğruluğu\"], rows,\n                 note=\"Aynı ölçekte CNN'in yerellik ve kaydırma eşdeğerliği önseli az veride belirgin üstünlük sağlar; veri arttıkça fark kapanır (makalenin ana bulgusunun minyatürü). Konum embedding'i olmadan ViT yamaların nerede olduğunu bilmez — bir 'yama torbası' (x2). [class] token'ı ile GAP benzer sonuç verir (makale Ek D.3; GAP farklı öğrenme oranı ister).\")\n    R_.test(\"CNN vs ViT\", t_train)\n",
 "e": "import time\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        conv = nn.Conv2d(3, 384, 16, 16)\n        x = torch.randn(8, 3, 224, 224)\n        W, b = conv.weight.flatten(1), conv.bias\n        with torch.no_grad():\n            ref = conv(x).flatten(2).transpose(1, 2)\n            close(need(ns, \"patch_embed_unfold\")(x, W, b, 16).numpy(), ref.numpy(), atol=1e-4, what=\"unfold\")\n            close(need(ns, \"patch_embed_reshape\")(x, W, b, 16).numpy(), ref.numpy(), atol=1e-4, what=\"reshape\")\n            tm = {}\n            for name, f in ((\"conv\", lambda: conv(x)), (\"unfold\", lambda: ns[\"patch_embed_unfold\"](x, W, b, 16)), (\"reshape\", lambda: ns[\"patch_embed_reshape\"](x, W, b, 16))):\n                f()\n                t0 = time.perf_counter()\n                for _ in range(3):\n                    f()\n                tm[name] = (time.perf_counter() - t0) / 3\n            rows = []\n            layer = nn.TransformerEncoderLayer(384, 6, 1536, 0.0, activation=\"gelu\", batch_first=True, norm_first=True).eval()\n            for P in (32, 16, 8):\n                T = (224 // P) ** 2 + 1\n                z = torch.randn(4, T, 384)\n                layer(z)\n                t0 = time.perf_counter()\n                layer(z)\n                tl = time.perf_counter() - t0\n                lin = 4 * T * 384 * 384 + 2 * T * 384 * 1536\n                att = 2 * T * T * 384\n                rows.append([P, T, fmt_time(tl), f\"{att / (lin + att):.0%}\"])\n        R_.note(\"Yama embedding'i (batch 8, 224², D = 384): \" + \", \".join(f\"{k} {fmt_time(v)}\" for k, v in tm.items()) + \". Üçü aynı hesap; reshape kopyasız görünüm + tek matmul.\")\n        R_.table(\"Bir ViT-S katmanı (D = 384, 6 baş), batch 4, 224 × 224 (CPU)\", [\"P\", \"token\", \"süre\", \"attention FLOP payı\"], rows,\n                 note=\"Yama boyutu yarıya inince token 4×, attention maliyeti 16× artar. P = 8'de attention baskın hale gelir: küçük yama = ince ayrıntı ama karesel bedel. Swin'in pencere attention'ı (x3) bu bedeli doğrusala indirir.\")\n    R_.test(\"yama embedding'i ve yama boyutu\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"interp_pos\", \"sincos_2d\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        pos = rng.normal(size=(14 * 14, 6))\n        close(need(ns, \"interp_pos\")(pos, 14), pos, atol=1e-12, what=\"aynı boyut → değişmez\")\n        lin = np.add.outer(np.arange(14.), 2 * np.arange(14.))[..., None] * np.ones(3)\n        up = ns[\"interp_pos\"](lin.reshape(-1, 3), 24).reshape(24, 24, 3)\n        tgt = (np.arange(24) + 0.5) * 14 / 24 - 0.5\n        inner = (tgt >= 0) & (tgt <= 13)\n        close(up[np.ix_(inner, inner)][..., 0], np.add.outer(tgt[inner], 2 * tgt[inner]), atol=1e-10, what=\"doğrusal alan tam interpole edilir\")\n        P = need(ns, \"sincos_2d\")(14, 64)\n        ensure(P.shape == (196, 64), \"şekil\")\n        S = P @ P.T\n        a, b = S[14 * 5 + 5], S[14 * 5 + 6]\n        ensure(np.argmax(a) == 14 * 5 + 5 and a[14 * 5 + 6] > a[14 * 9 + 9], \"komşu konumlar daha benzer\")\n        close(S[14 * 2 + 3, 14 * 2 + 5], S[14 * 8 + 7, 14 * 8 + 9], atol=1e-10, what=\"benzerlik yalnız göreli yer değiştirmeye bağlı\")\n        R_.note(\"ViT'te konum embedding'leri öğrenilir ve 2B yapı yalnız ince ayarda interpolasyonla kullanılır; öğrenilmiş embedding'ler yine de 2B ızgarayı kendiliğinden kodlar (makale Şekil 7 orta: aynı satır/sütundaki yamalar benzer). Sabit 2B sin-cos (MAE, DeiT-III) bunu baştan verir ve göreli yer değiştirmeye göre değişmezdir.\")\n    R_.test(\"konum embedding'leri\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rng = np.random.default_rng(0)\n        N, k, D = 16, 12, 8\n        patches = rng.normal(size=(N, k))\n        E, cls = rng.normal(size=(k, D)) / 3, rng.normal(size=D)\n        pos = rng.normal(size=(N + 1, D))\n        W = [rng.normal(size=(D, D)) / 3 for _ in range(3)]\n        perm = rng.permutation(N)\n        a = need(ns, \"vit_forward\")(patches, E, cls, None, W)\n        b = ns[\"vit_forward\"](patches[perm], E, cls, None, W)\n        close(b[0], a[0], atol=1e-12, what=\"konumsuz: [class] çıktısı yama sırasından bağımsız\")\n        close(b[1:], a[1:][perm], atol=1e-12, what=\"yama çıktıları aynı permütasyonla yer değiştirir\")\n        c = ns[\"vit_forward\"](patches, E, cls, pos, W)\n        d = ns[\"vit_forward\"](patches[perm], E, cls, pos, W)\n        ensure(np.abs(c[0] - d[0]).max() > 1e-3, \"konumlu: sıra fark eder\")\n        R_.note(\"Konum embedding'i olmayan öz-attention permütasyona eşdeğerdir: yamaları karıştırınca çıktılar aynı şekilde karışır, [class] (ve GAP) değişmez. Böyle bir ViT için bir yüz ile parçaları karıştırılmış yüz aynı görüntüdür — d aşamasındaki konumsuz ViT'in düşük doğruluğunun nedeni. CNN'de bu bilgi konvolüsyonun yerelliğiyle zaten vardır.\")\n    R_.test(\"permütasyon eşdeğerliği\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"window_partition\", \"window_reverse\", \"shifted_window_mask\", \"attn_cost\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        x = rng.normal(size=(8, 12, 3))\n        w = need(ns, \"window_partition\")(x, 4)\n        ensure(w.shape == (6, 16, 3), \"6 pencere × 16 token\")\n        close(w[1], x[0:4, 4:8].reshape(16, 3), what=\"pencere 1 = satırlar 0–3, sütunlar 4–7\")\n        close(need(ns, \"window_reverse\")(w, 4, 8, 12), x, what=\"ters\")\n        m = need(ns, \"shifted_window_mask\")(8, 8, 4, 2)\n        ensure(m.shape == (4, 16, 16), \"maske şekli\")\n        ensure(m[0].all(), \"sol üst pencere tek bölge\")\n        ensure(m[3].sum() == 4 * 16 + 4 * 16 + 0 or m[3].sum() == (4 + 4 + 4 + 4) * 4, \"sağ alt pencere 4 bölgeye bölünür (her biri 2 × 2 = 4 token)\")\n        ensure(m[1].sum() == 8 * 8 + 8 * 8, \"sağ üst pencere 2 bölge (8'er token)\")\n        rows = []\n        for H, C, M in ((56, 96, 7), (112, 96, 7), (224, 96, 7)):\n            g, wn = need(ns, \"attn_cost\")(H, H, C, M)\n            rows.append([f\"{H} × {H}\", f\"{g / 1e9:.1f}\", f\"{wn / 1e9:.2f}\", f\"{g / wn:.0f}×\"])\n        R_.table(\"Swin'in ilk aşaması (C = 96, M = 7): küresel vs pencere MSA (GMAC)\", [\"öznitelik haritası\", \"küresel\", \"pencere\", \"oran\"], rows,\n                 note=\"Küresel MSA token sayısının karesiyle, pencere MSA doğrusal büyür (Swin Denk. 1–2). Pencereler arası bilgi akışı için her ikinci blokta pencereler ⌊M/2⌋ kaydırılır; kaydırılmış görüntüde bir pencere, aslında bitişik olmayan bölgeleri içerebilir — maske yalnız aynı bölgedeki token'ların birbirine bakmasına izin verir, böylece pencere sayısı artmadan (döngüsel kaydırma) hesap korunur.\")\n    R_.test(\"pencere attention'ı\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_15_vit as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Yöntem:** Transformer token dizisi bekler. Görüntü $x\in\mathbb R^{H\times W\times C}$, $N=HW/P^2$ adet düzleştirilmiş yamaya bölünür: $x_p\in\mathbb R^{N\times(P^2C)}$; her yama öğrenilebilir doğrusal izdüşümle $D$ boyutlu "yama embedding'ine" çevrilir. BERT'in [CLS]'i gibi başa öğrenilebilir bir **[class] token'ı** eklenir; kodlayıcı çıkışındaki durumu görüntü temsilidir ve üstüne sınıflandırma başı (ön eğitimde tek gizli katmanlı MLP, ince ayarda tek doğrusal katman) konur. **Konum embedding'leri** (standart öğrenilebilir 1B; 2B farkındalıklı sürümler anlamlı kazanç vermedi) eklenir. Kodlayıcı: dönüşümlü çok başlı öz-attention ve MLP (GELU) blokları, her bloktan önce LayerNorm (Pre-LN), sonra artık bağlantı:
$$z_0=[x_{cls};x_p^1E;\dots;x_p^NE]+E_{pos},\quad z'_\ell=\mathrm{MSA}(\mathrm{LN}(z_{\ell-1}))+z_{\ell-1},\quad z_\ell=\mathrm{MLP}(\mathrm{LN}(z'_\ell))+z'_\ell,\quad y=\mathrm{LN}(z_L^0).$$

**Endüktif yanlılık:** CNN'de yerellik, 2B komşuluk ve kaydırma eşdeğerliği her katmana gömülüdür; ViT'te yalnız MLP katmanları yerel ve kaydırma eşdeğerlidir, öz-attention küreseldir. 2B komşuluk yapısı yalnız iki yerde kullanılır: yamalara bölerken ve ince ayarda konum embedding'lerini farklı çözünürlüğe uyarlarken. Bu yüzden ViT orta boy veride (ImageNet) benzer boyutlu ResNet'lerin birkaç puan gerisinde kalır; büyük ölçekli ön eğitimde (ImageNet-21k, JFT-300M) geçer. **Hibrit:** yamalar ham pikseller yerine bir CNN'in öznitelik haritasından (1 × 1 yamalar dahil) alınabilir.

**İnce ayar:** genellikle daha yüksek çözünürlükte yapılır; P sabit kaldığı için dizi uzar — önceden eğitilmiş konum embedding'leri orijinal görüntüdeki yerlerine göre 2B interpolasyonla uyarlanır. Modeller: ViT-B (D = 768, L = 12, 86M), ViT-L (1024, 24, 307M), ViT-H (1280, 32, 632M); "ViT-L/16" = 16 × 16 yamalı Large.

## 1 · Elle

<!-- çözüm -->

N = (224/16)² = 14² = 196; [class] ile 197. 384'te 24² = 576 (577 token); konum ızgarası 14 × 14 → 24 × 24. Attention skor matrisi $T^2$: $(577/197)^2=8.58$ — çözünürlük 1.71 kat, attention maliyeti 8.6 kat.

ViT-B/16: yama embedding'i $16\cdot16\cdot3\cdot768+768=590\,592$. Katman: MSA $4(768^2+768)=2\,362\,368$, MLP $768\cdot3072+3072+3072\cdot768+768=4\,722\,432$, iki LN $4\cdot768=3\,072$ → **7 087 872**. Toplam: 590 592 + 768 ([class]) + 197 · 768 (konum) + 12 · 7 087 872 + 1 536 (son LN) + 769 000 (baş) = **86 567 656** ≈ 86M.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yamalar**

- `n224`: 224 × 224, P = 16: yama sayısı N
- `seq`: [class] token'ıyla dizi uzunluğu
- `n384`: 384 × 384'te ince ayar, P = 16: N
- `grid`: Konum embedding ızgarası 14 × 14'ten kaç × kaç'a interpole edilir? (kenar)
- `cost`: Attention skor matrisi maliyet oranı (384 / 224, [class] dahil): $(577/197)^2$

**ViT-B/16: D = 768, L = 12, MLP 3072, 1000 sınıf**

- `pe`: Yama embedding'i (RGB, bias dahil)
- `layer`: Bir kodlayıcı katmanı (MSA + MLP + 2 LN)
- `total`: Toplam ([class], konum, son LN ve doğrusal baş dahil)

In [ ]:
cevaplar = dict(
    n224=None,
    seq=None,
    n384=None,
    grid=None,
    cost=None,
    pe=None,
    layer=None,
    total=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"n224":[196,0],"seq":[197,0],"n384":[576,0],"grid":[24,0],"cost":[8.578654435826742,0.000001],"pe":[590592,0],"layer":[7087872,0],"total":[86567656,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python.

- `n_patches(H, W, P)`.
- `vit_params(D, L, M, P, C, img, classes)` — yama embedding'i + [class] + konum + L katman + son LN + doğrusal baş.
- `attn_flops(T, D)` — bir öz-attention katmanının skor ve ağırlıklı toplam kısmı: $2T^2D$ çarpma-toplama (QKᵀ ve AV).
- `patchify(img, P)` — tek kanallı görüntü (liste listesi) → yama listesi, yamalar satır satır (row-major), her yama satır satır düzleştirilmiş.

In [ ]:
%%writefile stage_b.py
def n_patches(H, W, P):
    ...


def vit_params(D, L, M, P, C, img, classes):
    ...


def attn_flops(T, D):
    ...


def patchify(img, P):
    ...

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz.

- `patchify(x, P)` — `(C, H, W)` → `(N, P·P·C)`; yamalar row-major, yama içi sıra `(p1, p2, c)` (einops: `c (h p1) (w p2) -> (h w) (p1 p2 c)`).
- `unpatchify(z, P, C, H, W)` — tersi.
- `patch_embed_conv(x, Wc, b, P)` — `Wc (D, C, P, P)`, adım P'li konvolüsyon (`sliding_window_view` + `einsum`) → `(N, D)`; `patchify(x) @ E + b` ile aynı olmalı (E'yi Wc'den yeniden düzenleyerek).
- `conv_to_linear(Wc)` → `(P·P·C, D)`.
- `mean_attention_distance(A, g, P)` — `A (N, N)` yamalar arası attention (satırlar 1'e toplanır), `g × g` ızgara: her sorgu için piksel cinsinden ağırlıklı ortalama uzaklık, sonra sorgular üzerinden ortalama (makale Şekil 7).

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def patchify(x, P):
    ...


def unpatchify(z, P, C, H, W):
    ...


def patch_embed_conv(x, Wc, b, P):
    ...


def conv_to_linear(Wc):
    ...


def mean_attention_distance(A, g, P):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch. Veri: sklearn rakamları 16 × 16'ya büyütülmüş (P = 4 → 16 yama).

- `PatchEmbed(P, C, D)` — `nn.Conv2d(C, D, P, P)`; çıktı `(B, N, D)`. `forward_unfold(x)`: aynı ağırlıklarla `F.unfold` + matris çarpımı (eşitlik testi).
- `ViT(img=16, P=4, D=64, L=4, h=4, classes=10, pool="cls", pos=True)` — `nn.TransformerEncoderLayer(norm_first=True, activation="gelu", batch_first=True)`, MLP oranı 2, dropout 0.1; `pool="gap"` yama token'larının ortalaması; `pos=False` konum embedding'i yok.
- `train_eval(model, Xtr, ytr, Xte, yte, epochs)` — AdamW (1e-3, wd 0.05), batch 32.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits


def digits16():
    X, y = load_digits(return_X_y=True)
    X = torch.tensor(X, dtype=torch.float32).view(-1, 1, 8, 8) / 16
    return F.interpolate(X, size=16, mode="bilinear", align_corners=False), torch.tensor(y)


class PatchEmbed(nn.Module):
    def __init__(self, P, C, D):
        super().__init__()
        ...

    def forward(self, x):
        ...

    def forward_unfold(self, x):
        ...


class ViT(nn.Module):
    def __init__(self, img=16, P=4, D=64, L=4, h=4, classes=10, pool="cls", pos=True):
        super().__init__()
        ...

    def forward(self, x):
        ...


def train_eval(model, Xtr, ytr, Xte, yte, epochs):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `patch_embed_unfold(x, W, b, P)` ve `patch_embed_reshape(x, W, b, P)` (`view` + `permute`; kopyasız yeniden şekillendirme) — `W (D, C·P·P)`.
- Ölçüm: 224 × 224 görüntü, P ∈ {32, 16, 8}: dizi uzunluğu, bir ViT-S katmanının (D = 384, 6 baş) süresi ve attention payı.

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn.functional as F


def patch_embed_unfold(x, W, b, P):
    ...


def patch_embed_reshape(x, W, b, P):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Konum embedding'leri: interpolasyon ve 2B sin-cos

NumPy.

- `interp_pos(pos, new_g)` — `pos (g·g, D)` ızgara embedding'lerini `new_g × new_g`'ye bilineer interpolasyon (align_corners=False: hedef koordinat $(i+0.5)\,g/g'-0.5$, kenarlarda sabitle). Döngüsüz.
- `sincos_2d(g, D)` — MAE tarzı sabit 2B konumlar: ilk D/2 boyut satır koordinatının, son D/2 sütun koordinatının 1B sin-cos kodlaması.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def interp_pos(pos, new_g):
    ...


def sincos_2d(g, D):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Konumsuz ViT = yama torbası (permütasyon eşdeğerliği)

NumPy. Tek katmanlı ViT kodlayıcısı (tek baş, LN yok): `encoder(z, Wq, Wk, Wv)` = $z+\mathrm{softmax}(QK^\top/\sqrt D)V$.

- `vit_forward(patches, E, cls, pos, W)` — `patches (N, P²C)`; `pos=None` ise konum eklenmez; çıktı `(N + 1, D)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def encoder(z, Wq, Wk, Wv):
    s = (z @ Wq) @ (z @ Wk).T / np.sqrt(z.shape[1])
    a = np.exp(s - s.max(1, keepdims=True))
    return z + (a / a.sum(1, keepdims=True)) @ (z @ Wv)


def vit_forward(patches, E, cls, pos, W):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Swin: pencere attention'ı ve kaydırılmış pencereler

NumPy, döngüsüz.

- `window_partition(x, M)` — `(H, W, C)` → `(H/M · W/M, M·M, C)`; `window_reverse(w, M, H, W)`.
- `shifted_window_mask(H, W, M, s)` — `np.roll(-s)` ile kaydırılmış görüntüde her pencere için `(M², M²)` izin maskesi: yalnız aynı orijinal bölgeden (kaydırmadan önce bitişik olan) token'lar birbirine bakar (Swin, Şekil 4).
- `attn_cost(H, W, C, M)` → `(küresel, pencere)`: $4hwC^2+2(hw)^2C$ ve $4hwC^2+2M^2hwC$.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def window_partition(x, M):
    ...


def window_reverse(w, M, H, W):
    ...


def shifted_window_mask(H, W, M, s):
    ...


def attn_cost(H, W, C, M):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Dosovitskiy ve ark. (2021): ViT'i CIFAR-10'da sıfırdan

[Dosovitskiy ve ark. — An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (ICLR 2021)](https://arxiv.org/abs/2010.11929)

d aşamasındaki ViT'i CIFAR-10'a taşı (32 × 32, P = 4, D = 192, L = 6): (1) artırmasız, (2) RandAugment + Mixup/CutMix + etiket yumuşatma ile eğit; aynı bütçeli bir ResNet-18 ile karşılaştır. Makalenin analizlerini küçük ölçekte tekrarla: öğrenilmiş konum embedding'lerinin benzerlik haritası (Şekil 7 orta), katman başına ortalama attention uzaklığı (Şekil 7 sağ, c aşamasındaki fonksiyonla), yama boyutu P ∈ {2, 4, 8} için doğruluk/hesap ödünleşimi.


**Adımlar**

1. Ortalama attention uzaklığı: başlar boyunca ve 100 test görüntüsü üzerinde ortala.

**Kabul ölçütleri**

- Artırma ViT'e CNN'den çok daha fazla yarar (endüktif yanlılık eksikliğini veri artırma telafi eder).
- İlk katmanlarda bazı başlar yerel (kısa uzaklık), bazıları küresel; derinlikle uzaklık artar.

## Makale ödevi: Touvron ve ark. (2021): DeiT ve damıtma token'ı

[Touvron ve ark. — Training data-efficient image transformers & distillation through attention (ICML 2021)](https://arxiv.org/abs/2012.12877)

d aşamasındaki ViT'e [class] token'ının yanına bir damıtma token'ı ekle; öğretmen olarak eğitilmiş bir CNN kullanıp (d aşamasındaki ya da CIFAR'da ResNet) yumuşak (KL, sıcaklık τ) ve sert (öğretmenin argmax'ı) damıtmayı karşılaştır (makale Tablo 3). Çıkarımda iki token'ın başlarının ortalamasını kullan; iki token'ın vektörleri arasındaki kosinüs benzerliğinin katmanlar boyunca nasıl değiştiğini ölç.


**Adımlar**

1. Sert damıtma kaybı: ½ CE(Z_cls, y) + ½ CE(Z_dist, argmax öğretmen).

**Kabul ölçütleri**

- CNN öğretmenli sert damıtma, damıtmasız DeiT'i geçer.
- İki token'ın benzerliği derin katmanlarda artar ama 1'e ulaşmaz.

## Kaynaklar

- [Dosovitskiy ve ark. — An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale (ICLR 2021)](https://arxiv.org/abs/2010.11929) — arXiv
- [Touvron ve ark. — Training data-efficient image transformers & distillation through attention (DeiT, ICML 2021)](https://arxiv.org/abs/2012.12877) — arXiv
- [Liu ve ark. — Swin Transformer (ICCV 2021)](https://arxiv.org/abs/2103.14030) — arXiv
- [He ve ark. — Masked Autoencoders Are Scalable Vision Learners (CVPR 2022)](https://arxiv.org/abs/2111.06377) — arXiv
- [UvA DL Notebooks — Tutorial 15: Vision Transformers](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial15/Vision_Transformer.html) — MIT
- [d2l.ai — 11.8 Transformers for Vision](https://d2l.ai/chapter_attention-mechanisms-and-transformers/vision-transformer.html) — CC-BY-SA 4.0